# Logistic Regression
Partiamo dalla Logistic Regression con soft max. 
Pensiamo il problema posto come il seguente: "posso prevedere come sarà l'indice di qualità dell'aria tra un'ora attorno ad una data stazione, usando informazioni dell'arpav e dell'appa per le ore a questa precedenti?"

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn import datasets

from  sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import f1_score

In [22]:
# prepariamo il dataframe per questo studio (poi questa parte va probabilmente spostata)
dataset_df = pd.read_csv('../../data/processed/dataset_EDA_processed.csv')
feature_columns = ['station_appa', 'date', 'hour', 'week_day', 'elevation', 'CO', 'NO2', 'O3', 'PM10', 'PM2.5', 'SO2', 'tot_area_power', 'temperature', 'winds_spd', 'precipitation']
target = dataset_df['EAQI']

In [32]:
# vogliamo prevedere l'EAQI ad un'ora date le informazioni sulle ore precedenti
# per evitare di appesantire troppo il dataframe consideriamo solo le tre ore precedenti per tutti i dati eccetto le precipitazioni
# abbiamo visto una correlasione particolare per gli inquinanti con le precipitazioni dalle 5 ore prima in poi. Per questo dato prendiamo quindi le precipitazioni
# di 5, 6 e 7 ore prima (invece che delle tre ore appena precedenti)

feature_df = dataset_df[feature_columns[:5]].copy()
for h in range (1,4):
    feature_df[[col + f'_{h}' for col in feature_columns[5:13]]] = dataset_df.groupby('station_appa')[feature_columns[5:13]].shift(h)
    feature_df[f'precipitation_{4 + h}'] = dataset_df.groupby('station_appa')[feature_columns[-1]].shift(5 + h)

feature_df = feature_df.iloc[8:].reset_index(drop=True)
feature_df.head()

,station_appa,date,hour,week_day,elevation,CO_1,NO2_1,O3_1,PM10_1,PM2.5_1,...,precipitation_6,CO_3,NO2_3,O3_3,PM10_3,PM2.5_3,SO2_3,tot_area_power_3,temperature_3,precipitation_7
0,Borgo Valsugana,2013-11-02,8,5,410,NaN,16.0,10.0,23.0,NaN,...,0.0,NaN,11.0,15.0,24.0,19.0,NaN,939.767930,9.8,0.0
1,Borgo Valsugana,2013-11-02,9,5,410,NaN,21.0,6.0,24.0,NaN,...,0.0,NaN,11.0,13.0,23.0,19.0,NaN,934.069336,9.7,0.0
2,Borgo Valsugana,2013-11-02,10,5,410,NaN,19.0,10.0,29.0,NaN,...,0.0,NaN,16.0,10.0,23.0,NaN,NaN,927.923195,9.6,0.0
3,Borgo Valsugana,2013-11-02,11,5,410,NaN,23.0,10.0,29.0,NaN,...,0.0,NaN,21.0,6.0,24.0,NaN,NaN,921.967223,9.9,0.0
4,Borgo Valsugana,2013-11-02,12,5,410,NaN,23.0,13.0,27.0,NaN,...,0.0,NaN,19.0,10.0,29.0,NaN,NaN,916.103636,10.3,0.0
